In [1]:
import os
os.environ['HF_ENDPOINT'] = "https://hf-mirror.com"

## (1) Load Model & Weights from HuggingFace

In [2]:
from model import Mamba
from helper import load_from_cache, generate, prefill, step_fn

base_path="/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("load done")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


load done


In [3]:
def generate_demo(prompt, gen_len=10, seed=42):
    input_ids = tokenizer.encode(prompt, return_tensors='jax')
    output_ids = generate(model, params, input_ids, gen_len, seed=seed)
    print(prompt, tokenizer.decode(output_ids[0]), sep='')

In [4]:
generate_demo('Mamba is the')

Mamba is the highest mountain in the province, and also the highest


In [5]:
generate_demo('The meaning of life is ')

The meaning of life is ʻinērʻ (ʻ


In [6]:
generate_demo('def reverse_string(')

def reverse_string(self, prefix):
    """
    Helper


In [7]:
generate_demo('The meaning of life is ', seed=114514)

The meaning of life is 
"a state of being and the state of


## (2) SPU

### 2.1: 定义simulator

> 教程视频中用了一堆的`spu_pb2`，但现在的spu中根本没有这个模块，根据成员推测，尝试换成`libspu`

In [8]:
import spu.utils.simulation as spsim
import spu.libspu as libspu

sim = spsim.Simulator.simple(2,libspu.ProtocolKind.CHEETAH, libspu.FieldType.FM64)
# sim_aby = spsim.Simulator.simple(2,libspu.ProtocolKind.ABY3, libspu.FieldType.FM64)

In [9]:
# define cheetah config with pphlo trace and profile on
config_che = libspu.RuntimeConfig(
    protocol = libspu.ProtocolKind.CHEETAH,
    field = libspu.FieldType.FM64,
    fxp_fraction_bits = 18,
    # enable_pphlo_trace = True,
    # enable_pphlo_profile = True,
)

### 2.2: 定义运行函数

In [16]:
import jax
import jax.numpy as jnp
from functools import partial

# gen_spu = jax.jit(generate, static_argnames=['model','n_tokens_to_gen','sample','top_k'])

@partial(jax.jit, static_argnames=['n_tokens_to_gen','sample','top_k'])
def gen_spu(params, input_ids, n_tokens_to_gen: int = 5,
             sample: bool = True, top_k: int = 40, seed: int = 42):
    key = jax.random.PRNGKey(seed)
    next_token_logits, states = prefill(model, params, input_ids)

    generated = jnp.zeros((input_ids.shape[0], n_tokens_to_gen), dtype=input_ids.dtype)

    for i in range(n_tokens_to_gen):
        if top_k is not None:
            values, _ = jax.lax.top_k(next_token_logits, k=top_k)
            kth_values = values[:, -1:]
            next_token_logits = jnp.where(next_token_logits < kth_values, -1e9, next_token_logits)

        probs = jax.nn.softmax(next_token_logits, axis=-1)

        if sample:
            key, subkey = jax.random.split(key)
            next_id = jax.random.categorical(subkey, jnp.log(probs + 1e-9), axis=-1)
        else:
            next_id = jnp.argmax(probs, axis=-1)

        generated = generated.at[:, i].set(next_id)

        next_token_logits, states = step_fn(model, params, next_id, states)

    return generated

### 2.3: 运行密态程序

In [17]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
# output_ids = spsim.sim_jax(sim, gen_spu)(
#     model, params, input_ids, n_tokens_to_gen = 10, sample = False, top_k = None, seed = 42
# )
output_ids = gen_spu(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is in a sense a kind


In [ ]:
prompt = "Python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
output_ids = spsim.sim_jax(sim, gen_spu)(
    params, input_ids
)
# output_ids = gen_spu(model, params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

[2026-03-30 17:19:29.986] [info] [thread_pool.cc:30] Create a fixed thread pool with size 19
[2026-03-30 17:19:36.888] [info] [cheetah_mul.cc:335] CheetahMul uses 4 modulus for 64 bit input over 64 bit ring
[2026-03-30 17:19:36.889] [info] [cheetah_mul.cc:335] CheetahMul uses 4 modulus for 64 bit input over 64 bit ring
